In [ ]:
import torch
import torch.optim as optim
from dataloaders import load_data
from decompositions.vm_3d import VM3D
from decompositions.deep_tensor_3d import CP_DIP
from decompositions.SliceNet import SliceNet
from utilitys import save_loss_to_csv, plot_losses, create_output_dir, log_experiment_details
import time
import os

In [ ]:
# Load Data
data, input_size, data_name = load_data("skull") 
print(f"Input data shape: {data.shape}")
print(f"Input size (H, W, D): {input_size}")
# data.shape = (B, D, H, W)

Input data shape: torch.Size([1, 256, 68, 256])
Input size (H, W, D): (256, 68, 256)


In [ ]:
k=8

# base_channels=32
# model = SliceNet(k,base_channels)
# optimizer = optim.Adam(model.parameters(), lr=0.001)


In [90]:
# base_channels = 60
# model = CP_DIP(k=k,base_channels=base_channels,input_size=input_size)
# optimizer = optim.Adam(model.parameters(), lr=0.001)

In [91]:
# base_channels=25
# model = VM3D(k=k,base_channels=base_channels,input_size=input_size)
# optimizer = optim.Adam(model.parameters(), lr=0.001)

In [92]:
save_path, _ = create_output_dir(model.__class__.__name__, data_name, k)


Created output directory: outputs\CP_skull_k8_0923


In [ ]:
import torch

def train(model, data, optimizer, epochs, save_path, use_k_loop=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device)
    model.to(device).train()
    data = data.to(device)

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

    if use_k_loop:
        k_losses = {k: {'mse': [], 'ortho': [], 'total': []} for k in range(1, model.k + 1)}
        start_time = time.time()
        for k in range(1, model.k + 1):
            print(f"\nStart to train k={k}")
            for epoch in range(epochs):
                optimizer.zero_grad()
                output = model.forward(data, k)
                loss_recon, loss_ortho, total_loss = model.loss(data, output)

                k_losses[k]['mse'].append(loss_recon.item())
                k_losses[k]['ortho'].append(loss_ortho.item())
                k_losses[k]['total'].append(total_loss.item())

                total_loss.backward()
                optimizer.step()

                if epoch % 10 == 0:
                    print(f'Epoch: {epoch:3d}/{epochs}, '
                          f'MSE Loss: {loss_recon.item():.6f}, Ortho Loss: {loss_ortho.item():.6f}, '
                          f'Total Loss: {total_loss.item():.6f}')

        training_time = time.time() - start_time

        peak_mem = (torch.cuda.max_memory_allocated(device) / 1024**2 
                    if device.type == "cuda" else "N/A")

        results = {
            "model": model.__class__.__name__,
            "data": data_name,
            "k": model.k,
            "base_channels": getattr(model, 'base_channels', 'N/A'),
            "epochs": epochs,
            "optimizer": optimizer.__class__.__name__,
            "learning_rate": optimizer.param_groups[0]['lr'],
            "training_time": f'{training_time:.2f}s',
            "training_mode": "k_loop",
        }
        log_experiment_details(results, save_path)
        print(f"Everything is done! Saved to {save_path}, Peak Mem: {peak_mem} MB")
        return model, k_losses

    else:
        losses = {'mse': [], 'ortho': [], 'total': []}
        start_time = time.time()
        print(f"\nStart to train for {epochs} epochs")
        for epoch in range(epochs):
            optimizer.zero_grad()
            output = model.forward(data)
            loss_recon, loss_ortho, total_loss = model.loss(data, output)

            losses['mse'].append(loss_recon.item())
            losses['ortho'].append(loss_ortho.item())
            losses['total'].append(total_loss.item())

            total_loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                print(f'Epoch: {epoch:3d}/{epochs}, '
                      f'MSE Loss: {loss_recon.item():.6f}, Orthogonality Loss: {loss_ortho.item():.6f}, '
                      f'Total Loss: {total_loss.item():.6f}')

        training_time = time.time() - start_time

        peak_mem = (torch.cuda.max_memory_allocated(device) / 1024**2 
                    if device.type == "cuda" else "N/A")

        results = {
            "model": model.__class__.__name__,
            "data": data_name,
            "k": model.k,
            "base_channels": base_channels,
            "epochs": epochs,
            "optimizer": optimizer.__class__.__name__,
            "learning_rate": optimizer.param_groups[0]['lr'],
            "training_time": f'{training_time:.2f}s',
            "training_mode": "direct",
            "peak_memory_MB": peak_mem, 
        }
        log_experiment_details(results, save_path)
        print(f"All operations completed! Save path: {save_path}, Peak Mem: {peak_mem} MB")
        return model, losses


In [94]:
model,losses = train(model, data, optimizer, epochs=10, save_path=save_path,use_k_loop=0)

cuda

Start to train for 10 epochs
Epoch:   0/10, MSE Loss: 0.018687, Orthogonality Loss: 0.018687, Total Loss: 0.018687
Experiment details saved to: outputs\CP_skull_k8_0923\experiment_details.txt
All operations completed! Save path: outputs\CP_skull_k8_0923, Peak Mem: 883.4384765625 MB


In [ ]:
import tensorly as tl
from tensorly.decomposition import parafac, tucker
import ipyvolume as ipv
import ipywidgets as widgets
import numpy as np
import os
import matplotlib.pyplot as plt
import torch.nn.functional as F
tl.set_backend('pytorch')

def save_ipyvolume_html(filename, figure=None):
    if figure is None:
        figure = ipv.gcf()
    ipv.save(filename)
    print(f"[OK] Saved interactive HTML to: {filename}")

def save_mse_to_txt(mse_dict, save_path):
    """
    Save MSE results to a txt file
    
    Args:
    mse_dict: dictionary containing MSE values of different methods
    save_path: path to save the results
    """
    filepath = os.path.join(save_path, 'reconstruction_mse.txt')
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("Reconstruction MSE results:\n")
        f.write("=" * 30 + "\n")
        for method, mse in mse_dict.items():
            f.write(f"{method}: {mse:.6f}\n")
    print(f"[OK] Saved MSE results to: {filepath}")


def hosvd_decomposition(data, k):
    """
    HOSVD decomposition (via tensorly's tucker with truncated_svd)
    data: tensor [B, H, W, D]  (batch in front)
    k: rank (same for all modes)
    """
    B = data.shape[0]
    X_rec = torch.zeros_like(data)

    ranks = [k, k, k]

    factors_list = []
    cores_list = []

    for b in range(B):
        # Tucker with truncated_svd == HOSVD
        core, factors = tucker(
            data[b],
            rank=ranks,
            init='svd',          # use SVD init instead of random
            svd='truncated_svd'  # ensures truncated SVD per mode
        )

        X_rec[b] = tl.tucker_to_tensor((core, factors))
        factors_list.append(factors)
        cores_list.append(core)

    return {
        'X_rec': X_rec,
        'cores': cores_list,
        'factors': factors_list
    }


def cp_decomposition(data, k, restarts=2, max_iter=5000, tol=1e-7, verbose=False):
    """
    Simplified enhanced version of CP-ALS:
      - Optimizes on the original scale (no normalization)
      - init='svd' + a few random restarts
      - Larger iteration limit, stricter tolerance
      - Enable linesearch if available
    Return structure is the same as the original function 
    (weights/factors from the last batch as the best solution).
    """
    B = data.shape[0]
    device, dtype = data.device, data.dtype
    X_rec = torch.empty_like(data)

    def run_parafac(x, init='svd', seed=None):
        kwargs = dict(rank=k, n_iter_max=max_iter, tol=tol, init=init,
                      normalize_factors=False, verbose=False)
        if init == 'random':
            kwargs['random_state'] = seed
        # Try enabling linesearch (fallback if not supported)
        try:
            kwargs['linesearch'] = True
            return parafac(x, **kwargs)
        except TypeError:
            kwargs.pop('linesearch', None)
            return parafac(x, **kwargs)

    best_w, best_f = None, None  # Only keep the factors of the last batch (same as your original interface)

    for b in range(B):
        x = data[b]                         # [I,J,K] torch.Tensor
        x_tl = tl.tensor(x, device=device, dtype=dtype)

        # First try SVD init
        w, f = run_parafac(x_tl, init='svd')
        xr = tl.cp_to_tensor((w, f))
        best_mse = F.mse_loss(xr, x).item()
        best = (w, f)
        if verbose:
            print(f"[b={b}] svd  mse={best_mse:.6e}")

        # A few random restarts
        for s in range(restarts):
            wr, fr = run_parafac(x_tl, init='random', seed=s)
            xr = tl.cp_to_tensor((wr, fr))
            mse = F.mse_loss(xr, x).item()
            if verbose:
                print(f"[b={b}] rand{ s:02d} mse={mse:.6e}")
            if mse < best_mse:
                best_mse, best = mse, (wr, fr)

        # Record best result
        w, f = best
        X_rec[b] = tl.cp_to_tensor((w, f))
        best_w, best_f = w, f  # Only keep the last batch's factors

    return {
        'X_rec': X_rec,
        'weights': best_w,
        'factors': best_f,
    }

def visualize_3d_results(original_data, model_rec, k, threshold=0.0, save_path=None):
    # Traditional decompositions
    tucker_rec = hosvd_decomposition(original_data, k)['X_rec']
    cp_rec = cp_decomposition(original_data, k)['X_rec']

    original = original_data[0].detach().cpu().numpy()
    tucker = tucker_rec[0].detach().cpu().numpy()
    cp = cp_rec[0].detach().cpu().numpy()
    model = model_rec[0].detach().cpu().numpy()

    datas = [original, tucker, cp, model]
    titles = ['Original', 'Tucker', 'CP', 'Model']
    mse_errors = [np.mean((original - d) ** 2) for d in datas]
    mse_dict = {
        'Tucker': mse_errors[1],
        'CP': mse_errors[2],
        'Model': mse_errors[3]
    }
    if save_path:
        save_mse_to_txt(mse_dict, save_path)
    widgets_list = []

    for data, title, mse in zip(datas, titles, mse_errors):
        ipv.figure()
        
        # Threshold filtering: set voxels below threshold to 0 to avoid affecting visualization
        data_filtered = np.copy(data)
        data_filtered[data_filtered < threshold] = 0.0
        
        # Use volshow directly to display volumetric data
        ipv.volshow(data_filtered, level=threshold)
        
        # Set axis ranges (optional, volshow defaults to full volume display)
        ipv.xlim(0, data.shape[2])
        ipv.ylim(0, data.shape[1])
        ipv.zlim(0, data.shape[0])

        canvas = ipv.gcf()
        text = widgets.HTML(f"<b>{title}</b><br>MSE: {mse:.6f}")
        widgets_list.append(widgets.VBox([text, canvas]))

        if save_path:
            os.makedirs(save_path, exist_ok=True)
            filepath = os.path.join(save_path, f"3d_{title.lower()}.html")
            save_ipyvolume_html(filepath, ipv.gcf())
            print(f"Saved: {filepath}")

    return widgets.HBox(widgets_list)


def ortho_vis(output, path):
    # Visualize orthogonality of model U matrices
    plt.figure(figsize=(15, 5))
    
    # Orthogonality of model U matrix
    with torch.no_grad():
        # Handle shape differences in model outputs
        U = output['U1']
        V = output['U2']
        W = output['U3']
        
        # If U and V are 3D tensors, take the first sample
        if U.dim() == 3:
            U = U[0]
        if V.dim() == 3:
            V = V[0]
        if W.dim() == 3:
            W = W[0]        
        
        U_norm = U / torch.norm(U, dim=0, keepdim=True)
        U_ortho = torch.matmul(U_norm.t(), U_norm)
        plt.subplot(131)  
        plt.imshow(U_ortho.cpu().detach().numpy(), cmap='viridis_r', vmin=-1, vmax=1)
        plt.colorbar()
        plt.title('Model U Orthogonality')
        
        # Orthogonality of model V matrix (cosine similarity matrix)
        V_norm = V / torch.norm(V, dim=0, keepdim=True)
        V_ortho = torch.matmul(V_norm.t(), V_norm)
        plt.subplot(132)  
        plt.imshow(V_ortho.cpu().detach().numpy(), cmap='viridis_r', vmin=-1, vmax=1)
        plt.colorbar()
        plt.title('Model V Orthogonality')
        
        plt.tight_layout()

        W_norm = W / torch.norm(W, dim=0, keepdim=True)
        W_ortho = torch.matmul(W_norm.t(), W_norm)
        plt.subplot(133)  
        plt.imshow(W_ortho.cpu().detach().numpy(), cmap='viridis_r', vmin=-1, vmax=1)
        plt.colorbar()
        plt.title('Model W Orthogonality')
        
        plt.tight_layout()

    plt.savefig(os.path.join(path, 'orthogonality_comparison.png'))
    plt.close()
def visualize_wrapper_3d(data, model, k, losses, save_path):
    """
    3D visualization wrapper function, compatible with two loss data formats
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Execute visualization
    fig = visualize_3d_results(
        original_data=data,
        model_rec=model.forward(data.to(device))['X_rec'],
        k=k,
        save_path=save_path
    )
    return fig

# Visualize results
fig = visualize_wrapper_3d(data, model, k, losses, save_path)


In [ ]:
fig